In [0]:
%sql
CREATE TABLE IF NOT EXISTS helathcare_audit.audit_table.Patient_load_details
(
    run_id STRING,
    record_type STRING,
    notebook_name STRING,
    layer STRING,
    table_name STRING,
    record_count BIGINT,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration_seconds BIGINT,
    status STRING,
    duplicate_check_status STRING,
    primary_key_status STRING,
    null_check_status STRING,
    standardization_status STRING,
    error_message STRING,
    load_date DATE
)
USING DELTA;

In [0]:
 %sql
select * from  helathcare_audit.audit_table.Patient_load_details 

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    TimestampType
)

# ==========================================
# Get Notebook Metadata
# ==========================================

def get_audit_metadata(target_table):

    notebook_name = (
        dbutils.notebook.entry_point
        .getDbutils()
        .notebook()
        .getContext()
        .notebookPath()
        .get()
        .split("/")[-1]
    )

    if target_table == "WORKFLOW":

        return (
            "Workflow_Summary",
            "Workflow",
            "Workflow"
        )

    table_name = target_table.split(".")[-1]

    catalog_name = target_table.split(".")[0]

    if "bronze" in catalog_name.lower():
        layer = "Bronze"

    elif "silver" in catalog_name.lower():
        layer = "Silver"

    elif "gold" in catalog_name.lower():
        layer = "Gold"

    else:
        layer = "Unknown"

    return notebook_name, table_name, layer


# ==========================================
# Write Audit
# ==========================================

def write_audit(
    target_table,
    run_id,
    record_count,
    start_time,
    end_time,
    status,
    duplicate_check_status=None,
    primary_key_status=None,
    null_check_status=None,
    standardization_status=None,
    error_message=None,
    record_type="NOTEBOOK"
):

    notebook_name, table_name, layer = get_audit_metadata(
        target_table
    )

    duration_seconds = int(
        (end_time - start_time).total_seconds()
    )

    audit_data = [(
        str(run_id),
        record_type,
        notebook_name,
        layer,
        table_name,
        record_count if record_count else 0,
        start_time,
        end_time,
        duration_seconds,
        status,
        duplicate_check_status,
        primary_key_status,
        null_check_status,
        standardization_status,
        error_message
    )]

    audit_schema = StructType([

        StructField("run_id", StringType(), True),

        StructField("record_type", StringType(), True),

        StructField("notebook_name", StringType(), True),

        StructField("layer", StringType(), True),

        StructField("table_name", StringType(), True),

        StructField("record_count", LongType(), True),

        StructField("start_time", TimestampType(), True),

        StructField("end_time", TimestampType(), True),

        StructField("duration_seconds", LongType(), True),

        StructField("status", StringType(), True),
        StructField("duplicate_check_status", StringType(), True),
        StructField("primary_key_status", StringType(), True),
        StructField("null_check_status", StringType(), True),
        StructField("standardization_status", StringType(), True),

        StructField("error_message", StringType(), True)

    ])

    audit_df = spark.createDataFrame(
        audit_data,
        schema=audit_schema
    )

    audit_df = audit_df.withColumn(
        "load_date",
        F.current_date()
    )

    try:

        audit_df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(
                "helathcare_audit.audit_table.Patient_load_details"
            )

    except Exception as e:

        print(
            f"Audit Write Failed: {str(e)}"
        )